In [0]:
scored = spark.read.table("workspace.default.lc_test_scored")   # has PD per loan
print("Scored test rows:", scored.count())
print(scored.columns)

Scored test rows: 225639
['is_bad', 'probability', 'inc_bin_woe', 'dti_bin_woe', 'grade_num_woe', 'home_ownership_idx_woe', 'purpose_idx_woe', 'verification_status_idx_woe', 'term_months_woe']


In [0]:
for t in ["lc_test_scored", "lc_test_woe", "lc_features"]:
    cols = spark.table(f"workspace.default.{t}").columns
    print(t, "→", "HAS id" if "id" in cols else "NO id", "|", len(cols), "cols")

lc_test_scored → NO id | 9 cols
lc_test_woe → NO id | 10 cols
lc_features → NO id | 20 cols


In [0]:
df = df.withColumn("EAD", col("funded_amnt"))

In [0]:
from pyspark.sql.functions import col, regexp_extract, expr

raw_keys = spark.read.option("mode", "PERMISSIVE").csv(
    "/Volumes/workspace/default/lending_club/Lending_Club_Accepted.csv",
    header=True, inferSchema=False
).select("id", "issue_d", "funded_amnt", "recoveries", "total_rec_prncp") \
 .withColumn("funded_amnt",     col("funded_amnt").cast("double")) \
 .withColumn("recoveries",      col("recoveries").cast("double")) \
 .withColumn("total_rec_prncp", col("total_rec_prncp").cast("double")) \
 .withColumn("issue_year", expr("try_cast(regexp_extract(issue_d, '-([0-9]{4})$', 1) AS INT)"))
 

analytical = spark.table("workspace.default.lc_analytical")
df = analytical.join(raw_keys.drop("issue_d"), on="id", how="left")

test_anchor = df.filter(col("issue_year") >= 2017).filter(col("is_bad").isNotNull())

print("Test anchor rows:", test_anchor.count())
test_anchor.select("id", "funded_amnt", "int_rate",
                   "total_rec_prncp", "recoveries",
                   "loan_status", "issue_year").show(5)

Test anchor rows: 225639
+--------+-----------+--------+---------------+----------+-----------+----------+
|      id|funded_amnt|int_rate|total_rec_prncp|recoveries|loan_status|issue_year|
+--------+-----------+--------+---------------+----------+-----------+----------+
|97317393|    18000.0|   23.99|        18000.0|       0.0| Fully Paid|      2017|
|98151688|    11000.0|   14.99|        11000.0|       0.0| Fully Paid|      2017|
|97289926|    18000.0|   17.99|        18000.0|       0.0| Fully Paid|      2017|
|97386587|    28575.0|   13.49|        4288.73|   3386.14|Charged Off|      2017|
|97377528|    10000.0|   12.74|        1658.75|       0.0|Charged Off|      2017|
+--------+-----------+--------+---------------+----------+-----------+----------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col, regexp_extract, expr

raw_keys = spark.read.option("mode", "PERMISSIVE").csv(
    "/Volumes/workspace/default/lending_club/Lending_Club_Accepted.csv",
    header=True, inferSchema=False
).selectExpr(
    "id",
    "issue_d",
    "try_cast(funded_amnt     AS DOUBLE) AS funded_amnt",
    "try_cast(recoveries      AS DOUBLE) AS recoveries",
    "try_cast(total_rec_prncp AS DOUBLE) AS total_rec_prncp"
).withColumn(
    "issue_year",
    expr("try_cast(regexp_extract(issue_d, '-([0-9]{4})$', 1) AS INT)")
).drop("issue_d")

analytical = spark.table("workspace.default.lc_analytical")
df = analytical.join(raw_keys, on="id", how="left")

test_anchor = df.filter(col("issue_year") >= 2017).filter(col("is_bad").isNotNull())

print("Test anchor rows:", test_anchor.count())
test_anchor.select("id", "funded_amnt", "int_rate",
                   "total_rec_prncp", "recoveries",
                   "loan_status", "issue_year").show(5)

Test anchor rows: 225639
+--------+-----------+--------+---------------+----------+-----------+----------+
|      id|funded_amnt|int_rate|total_rec_prncp|recoveries|loan_status|issue_year|
+--------+-----------+--------+---------------+----------+-----------+----------+
|97317393|    18000.0|   23.99|        18000.0|       0.0| Fully Paid|      2017|
|98151688|    11000.0|   14.99|        11000.0|       0.0| Fully Paid|      2017|
|97289926|    18000.0|   17.99|        18000.0|       0.0| Fully Paid|      2017|
|97386587|    28575.0|   13.49|        4288.73|   3386.14|Charged Off|      2017|
|97377528|    10000.0|   12.74|        1658.75|       0.0|Charged Off|      2017|
+--------+-----------+--------+---------------+----------+-----------+----------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col, when, avg

co = df.filter(col("loan_status") == "Charged Off")

# Outstanding principal at default = funded - principal already repaid
co = co.withColumn("ead_at_default", col("funded_amnt") - col("total_rec_prncp"))

# Loss = what was outstanding minus what we clawed back post-default
co = co.withColumn("loss", col("ead_at_default") - col("recoveries"))

# LGD = loss / exposure at default, floored at 0 and capped at 1
co = co.withColumn("lgd_realised",
    when(col("ead_at_default") <= 0, 0.0)
    .otherwise((col("loss") / col("ead_at_default")))
)
co = co.withColumn("lgd_realised",
    when(col("lgd_realised") < 0, 0.0)
    .when(col("lgd_realised") > 1, 1.0)
    .otherwise(col("lgd_realised"))
)

portfolio_lgd = co.agg(avg("lgd_realised")).collect()[0][0]
print("Realised portfolio LGD:", round(portfolio_lgd, 4))

Realised portfolio LGD: 0.8903


In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

+--------+-------------------+-----------+
|database|tableName          |isTemporary|
+--------+-------------------+-----------+
|default |lc_analytical      |false      |
|default |lc_features        |false      |
|default |lc_scorecard_points|false      |
|default |lc_test_scored     |false      |
|default |lc_test_woe        |false      |
|default |lc_train_woe       |false      |
+--------+-------------------+-----------+



In [0]:
from pyspark.sql.functions import col, when, regexp_extract, expr, log as Flog, sum as Fsum, count as Fcount
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.sql.functions import expr

# ── A. Raw financial cols (PERMISSIVE + try_cast handles the corrupt rows)
raw_keys = spark.read.option("mode","PERMISSIVE").csv(
    "/Volumes/workspace/default/lending_club/Lending_Club_Accepted.csv",
    header=True, inferSchema=False
).selectExpr(
    "id",
    "try_cast(funded_amnt     AS DOUBLE) AS funded_amnt",
    "try_cast(recoveries      AS DOUBLE) AS recoveries",
    "try_cast(total_rec_prncp AS DOUBLE) AS total_rec_prncp",
    "issue_d"
).withColumn("issue_year",
    expr("try_cast(regexp_extract(issue_d, '-([0-9]{4})$', 1) AS INT)")).drop("issue_d")

df = spark.table("workspace.default.lc_analytical").join(raw_keys, on="id", how="left")
df = df.withColumn("annual_inc",  expr("try_cast(annual_inc  AS DOUBLE)")) \
       .withColumn("dti",         expr("try_cast(dti         AS DOUBLE)")) \
       .withColumn("term_months", expr("try_cast(term_months AS DOUBLE)")) \
       .withColumn("is_bad",      expr("try_cast(is_bad      AS INT)"))


# ── B. Split
train_df = df.filter((col("issue_year") <= 2016) & col("is_bad").isNotNull())
test_df  = df.filter((col("issue_year") >= 2017) & col("is_bad").isNotNull())

# ── C. Features (grade ordinal, income/dti bins, categorical indexing)
def add_grade(s):
    return s.withColumn("grade_num",
        when(col("grade")=="A",1.0).when(col("grade")=="B",2.0).when(col("grade")=="C",3.0)
        .when(col("grade")=="D",4.0).when(col("grade")=="E",5.0).when(col("grade")=="F",6.0)
        .otherwise(7.0))
train_df, test_df = add_grade(train_df), add_grade(test_df)

q_inc = train_df.approxQuantile("annual_inc",[0.2,0.4,0.6,0.8],0.02)
q_dti = train_df.approxQuantile("dti",[0.2,0.4,0.6,0.8],0.02)
def add_bins(s):
    return s.withColumn("inc_bin",
        when(col("annual_inc").isNull(),2.0).when(col("annual_inc")<=q_inc[0],0.0)
        .when(col("annual_inc")<=q_inc[1],1.0).when(col("annual_inc")<=q_inc[2],2.0)
        .when(col("annual_inc")<=q_inc[3],3.0).otherwise(4.0)) \
     .withColumn("dti_bin",
        when(col("dti").isNull(),2.0).when(col("dti")<=q_dti[0],0.0)
        .when(col("dti")<=q_dti[1],1.0).when(col("dti")<=q_dti[2],2.0)
        .when(col("dti")<=q_dti[3],3.0).otherwise(4.0))
train_df, test_df = add_bins(train_df), add_bins(test_df)

for rc, ic in [("home_ownership","home_ownership_idx"),
               ("verification_status","verification_status_idx"),("purpose","purpose_idx")]:
    ix = StringIndexer(inputCol=rc, outputCol=ic, handleInvalid="keep").fit(train_df)
    train_df, test_df = ix.transform(train_df), ix.transform(test_df)

# ── D. WoE via JOIN (no UDFs)
bin_cols = ["grade_num","term_months","inc_bin","dti_bin",
            "home_ownership_idx","verification_status_idx","purpose_idx"]
def woe_tbl(s, c):
    a = s.groupBy(c).agg(Fsum("is_bad").alias("bad"),(Fcount("*")-Fsum("is_bad")).alias("good"))
    tb, tg = a.agg(Fsum("bad")).collect()[0][0], a.agg(Fsum("good")).collect()[0][0]
    return a.withColumn("pct_bad",(col("bad")+0.5)/(tb+0.5)) \
            .withColumn("pct_good",(col("good")+0.5)/(tg+0.5)) \
            .withColumn(f"{c}_woe",Flog(col("pct_good")/col("pct_bad"))).select(c,f"{c}_woe")
for c in bin_cols:
    w = woe_tbl(train_df, c)
    train_df, test_df = train_df.join(w,on=c,how="left"), test_df.join(w,on=c,how="left")


asm = VectorAssembler(inputCols=[f"{c}_woe" for c in bin_cols], outputCol="features")
model_lr = LogisticRegression(featuresCol="features", labelCol="is_bad", maxIter=50) \
               .fit(asm.transform(train_df))
model_lr.transform(asm.transform(test_df)) \
    .withColumn("pd_prediction", vector_to_array(col("probability"))[1]) \
    .select("id","pd_prediction","is_bad","funded_amnt","int_rate",
            "total_rec_prncp","recoveries","loan_status","grade") \
    .write.mode("overwrite").saveAsTable("workspace.default.lc_test_scored_final")
print("Saved lc_test_scored_final")

Saved lc_test_scored_final


In [0]:
from pyspark.sql.functions import col
print("Null annual_inc:", df.filter(col("annual_inc").isNull()).count())
print("Null is_bad:    ", df.filter(col("is_bad").isNull()).count())

Null annual_inc: 0
Null is_bad:     0


In [0]:
from pyspark.sql.functions import col, sum as Fsum, avg as Favg

LGD = 0.8903
s = spark.read.table("workspace.default.lc_test_scored_final") \
         .withColumn("EAD", col("funded_amnt")) \
         .withColumn("EL",  col("pd_prediction") * LGD * col("funded_amnt"))

total_el  = s.agg(Fsum("EL")).collect()[0][0]
total_ead = s.agg(Fsum("EAD")).collect()[0][0]
print(f"Total expected loss: ${total_el:,.0f}")
print(f"EL rate:             {total_el/total_ead:.2%}")

actual = s.filter(col("loan_status")=="Charged Off") \
    .withColumn("realised_loss", (col("funded_amnt")-col("total_rec_prncp"))-col("recoveries")) \
    .agg(Fsum("realised_loss")).collect()[0][0]
print(f"\nPredicted EL: ${total_el:,.0f}")
print(f"Actual loss:  ${actual:,.0f}")
print(f"Pred/Actual:  {total_el/actual:.2f}")

s.groupBy("grade").agg(Favg("pd_prediction").alias("avg_pd"),
                       Fsum("EL").alias("total_el")).orderBy("grade").show()

Total expected loss: $598,114,433
EL rate:             18.35%

Predicted EL: $598,114,433
Actual loss:  $608,138,965
Pred/Actual:  0.98
+-----+--------------------+--------------------+
|grade|              avg_pd|            total_el|
+-----+--------------------+--------------------+
|    A|0.061441903280559186| 2.815108460998197E7|
|    B| 0.12983070189291648| 9.779024101151705E7|
|    C| 0.21681994044935818|2.0271397587756464E8|
|    D|  0.2888251825961888|1.3949039489179307E8|
|    E|  0.3727230118823093| 7.500613233574162E7|
|    F|  0.4591173533189259| 3.400508205498627E7|
|    G| 0.49942625863749546|2.0957521990669623E7|
+-----+--------------------+--------------------+



In [0]:
g